# 02 — Input bit-depth sweep (PyTorch)

Sweeps input quantization in [0,2,4,8] with fixed network precision.

In [2]:
from pathlib import Path
import sys, os

# ---- Path setup (adjust if your repo layout differs) ----
PROJECT_ROOT = Path("..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [4]:
import pandas as pd

from config import ExperimentConfig, with_overrides
from runner import run_experiment
from utils import load_runs, flatten_runs
from plots import (
    plot_metric_vs_input_bits,
    plot_delta_from_baseline,
    plot_tradeoff_with_pareto,
)

pd.set_option("display.max_columns", 200)


In [5]:

base = ExperimentConfig(
    backend="pytorch",
    device="cuda",
    batch_size=256,
    model_precision="fp32",      
    input_quant_bits=0,        
    seed=42,
)

# choose your sweep set
in_bits_list = [0, 8, 4, 2, 1]   # remove 1 if you don't want it

cfgs = [with_overrides(base, input_quant_bits=b) for b in in_bits_list]

# cfgs += [with_overrides(base, model_precision="fp16", input_quant_bits=b) for b in in_bits_list]

ValueError: input_quant_bits must be one of (8,4,2), got 1

In [ ]:
records = []
for cfg in cfgs:
    payload, _ = run_experiment(cfg, split="val", save_results_flag=True)
    records.append(payload)

for r in records:
    print(r["run_id"], r["status"], r["results"].get("top1_acc"))

In [ ]:
runs = load_runs("./runs", status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

# Filter only what this notebook is responsible for
df_sweep = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.device"] == "cuda")
    & (df["cfg.model_precision"].isin(["fp32"]))   # include "fp16" if you enabled it
    & (df["cfg.input_quant_bits"].isin([0, 8, 4, 2, 1]))
].copy()

df_sweep[[
    "run_id",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.infer_ms_avg",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values(["cfg.model_precision", "cfg.input_quant_bits"])

In [ ]:
rows_sweep = df_sweep.to_dict(orient="records")

plot_metric_vs_input_bits(
    rows_sweep,
    metric_key="res.top1_acc",
    title="PyTorch: Top-1 vs input bit-depth",
    connect_points=False,
)

plot_delta_from_baseline(
    rows_sweep,
    baseline_selector={
        "cfg.backend": "pytorch",
        "cfg.model_precision": "fp32",
        "cfg.input_quant_bits": 0,
        "cfg.device": "cuda",
    },
    title="Input quantization sweep vs FP32 baseline",
)

plot_tradeoff_with_pareto(
    rows_sweep,
    x_key="res.infer_ms_avg",
    y_key="res.top1_acc",
    title="Input quantization: Accuracy–Latency tradeoff (Pareto)",
)

NameError: name 'df_sweep' is not defined